In [1]:
import os
import re
import datetime
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from time import sleep

In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'DZ BAL'

print(f"Running {regulatorName} Web Scraping Tool v.1.0")
now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(':', '.')[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder = os.path.dirname(os.path.abspath(__file__)) ## production
os.chdir(scriptfolder)

tempfolder = os.path.join(scriptfolder, 'tempfolder')
if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

Running DZ BAL Web Scraping Tool v.1.0


In [3]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
chromeOptions = webdriver.ChromeOptions()
prefs = {
    'plugins.always_open_pdf_externally': True,
    'download.prompt_for_download': False,
    'download.default_directory': tempfolder,
    'profile.default_content_setting_values.automatic_downloads': 1,
}
chromeOptions.add_experimental_option('prefs', prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()

In [4]:
#------------------------------------------------ Begin_Variable ----------------------------------------
sqldict = {
    'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [],
    'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 'InternalID_2_type': [],
    'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [],
    'Address_1': [], 'Address_2': [], 'City': [], 'Zip': [], 'Cntry': [],
    'Phone': [], 'Fax': [], 'Website': [], 'Email': [],
    'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],
    'RegCtry': [], 'RegCode': [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [],
    'ListName': [], 'ListProcessDate': [],
    'LEI Code': [], 'BIC SWIFT Code': [],
    'Name - Mother Company': [], 'Address_1 - Mother company': [], 'Address_2 -  Mother company': [],
    'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],
    'Phone - Mother company': [], 'Check': [],
}

processdate = now.strftime('%Y-%m-%d')

# Single list per requirements: "Banques Et Etablissements Financiers".
# regdict holds the parent (list-label) URL; the actual entity pages are the 5 category subpages
# walked through inside the main loop.
regdict = {
    regulatorName + ' 1': 'https://www.bank-of-algeria.dz/banques-et-etablissements-financiers/',
}

Typology = {
    regulatorName + ' 1': 'Banques Et Etablissements Financiers',
}

# Each tuple: (category page url, CoType label applied to entities found there)
CATEGORY_PAGES = [
    ('https://www.bank-of-algeria.dz/banques-commerciales-2/',                                    'Commercial Bank'),
    ('https://www.bank-of-algeria.dz/etablissement-financiers-a-vocation-generale/',              'Financial Establishment - General Purpose'),
    ('https://www.bank-of-algeria.dz/etablissements-financiers-a-vocation-specifique/',           'Financial Establishment - Specific Purpose'),
    ('https://www.bank-of-algeria.dz/bureaux-de-representation/',                                 'Representation Office'),
    ('https://www.bank-of-algeria.dz/association-des-banques-et-des-etablissements-financiers-abef/', 'Banking Association'),
]

In [5]:
#------------------------------------------------ Begin_Function ----------------------------------------
def bourange_same_length_array(sqldict):
    maxlen = len(sqldict['ListProcessDate'])
    for key in sqldict:
        if len(sqldict[key]) != maxlen:
            sqldict[key] += [''] * (maxlen - len(sqldict[key]))
    return sqldict

# Field labels seen on Bank of Algeria entity blocks (FR; site has no language toggle).
# Each entity <p> uses pattern: `<Label> : <value><br/>` repeated, but some entities use
# `<Label> – <value>` (em-dash) instead of a colon. BS4's get_text('\n') can also split
# label and value across lines, so we parse the whole text body by label-span (value of
# label_i runs until label_{i+1}) instead of per-line.
LABEL_RE = re.compile(
    r'(Si[èe]ge\s*Social|Adresse|'
    r'T[ée]l[ée]phone|DGA|Agence[^:–\n]{0,40}|'
    r'Fax|'
    r'Pr[ée]sident\s*Directeur\s*G[ée]n[ée]ral|Directeur\s*G[ée]n[ée]ral|Directrice\s*G[ée]n[ée]rale|'
    r'Pr[ée]sident\s*du\s*Directoire|Directeur\s*Ex[ée]cutif|Repr[ée]sentante?)'
    r'\s*[:–]\s*',
    re.IGNORECASE,
)
LABEL_FIELD = (
    ('address',  re.compile(r'^(Si[èe]ge\s*Social|Adresse)$', re.IGNORECASE)),
    ('phone',    re.compile(r'^(T[ée]l[ée]phone|DGA|Agence)', re.IGNORECASE)),
    ('fax',      re.compile(r'^Fax$', re.IGNORECASE)),
    ('director', re.compile(r'^(Directeur|Directrice|Pr[ée]sident|Repr[ée]sentante?)', re.IGNORECASE)),
)
ENTITY_TEXT_MARKERS = ('Téléphone', 'Telephone', 'Siège Social', 'Siege Social', 'Adresse', 'Fax')


def _classify(label):
    label = label.strip()
    for field, pat in LABEL_FIELD:
        if pat.match(label):
            return field
    return None


def parse_detail_block(p_text):
    """Parse one <p> entity-detail block.
    Find every label match in the text; the value of label_i is the substring from end-of-label_i
    to start-of-label_{i+1}. Phone aggregates Téléphone + DGA + Agence; fax aggregates Fax;
    director takes the first directeur/directrice/président/représentant(e).
    """
    text = p_text.replace('\xa0', ' ')
    parsed = {'address': '', 'phone': [], 'fax': [], 'director': ''}
    matches = list(LABEL_RE.finditer(text))
    if not matches:
        return {'address': '', 'phone': '', 'fax': '', 'director': ''}
    for i, m in enumerate(matches):
        label = m.group(1)
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        value = text[start:end].strip(' \t\n\r–-,;')
        value = re.sub(r'\s+', ' ', value)
        if not value:
            continue
        field = _classify(label)
        if field == 'address':
            if not parsed['address']:
                parsed['address'] = value
        elif field == 'phone':
            parsed['phone'].append(value)
        elif field == 'fax':
            parsed['fax'].append(value)
        elif field == 'director' and not parsed['director']:
            parsed['director'] = value
    parsed['phone'] = ' / '.join(parsed['phone'])
    parsed['fax'] = ' / '.join(parsed['fax'])
    return parsed


def split_address(addr):
    """Best-effort split of a French Algerian address.
    Country = DZ (entire site is Bank of Algeria), City defaults to 'Alger' when the
    address ends in 'Alger' or contains an 'Alger' segment; Zip extracted only when a
    5-digit code is present (most addresses don't include one).
    """
    addr = addr.replace('\xa0', ' ').strip(' ,–-')
    if not addr:
        return {'address_1': '', 'address_2': '', 'city': '', 'zip': '', 'cntry': 'DZ'}
    zip_match = re.search(r'\b(\d{5})\b', addr)
    zip_code = zip_match.group(1) if zip_match else ''
    city = ''
    for p in [s.strip() for s in re.split(r'[–-]', addr) if s.strip()][::-1]:
        if re.search(r'\bAlger\b', p, re.IGNORECASE):
            city = 'Alger'
            break
    return {
        'address_1': addr[:200],
        'address_2': addr[200:400] if len(addr) > 200 else '',
        'city': city,
        'zip': zip_code,
        'cntry': 'DZ',
    }


def extract_entities_from_page(html, category_url):
    """Yield (name, parsed_fields) tuples from a category page.
    Bank-of-Algeria pages use Elementor: each entity is a heading widget (h2) followed
    by a text-editor widget (p) inside the same `elementor-widget-wrap`. We filter to
    h2's whose sibling text-editor <p> contains at least one entity-field marker — this
    excludes page-title and footer headings (Rubriques, Contact, Suivez nous, copyright).
    """
    soup = BeautifulSoup(html, 'html.parser')
    results = []
    for heading in soup.select('h2.elementor-heading-title'):
        wrap = heading.find_parent(class_='elementor-widget-wrap')
        if wrap is None:
            wrap = heading.find_parent(class_=re.compile(r'elementor-(column|widget-wrap|element-populated)'))
        if wrap is None:
            continue
        text_widget = wrap.find(class_='elementor-widget-text-editor')
        if text_widget is None:
            continue
        p_el = text_widget.find('p')
        if p_el is None:
            continue
        block_text = p_el.get_text('\n').strip()
        if not any(marker.lower() in block_text.lower() for marker in ENTITY_TEXT_MARKERS):
            continue
        name = heading.get_text(' ', strip=True).replace('“', '"').replace('”', '"').strip()
        if not name:
            continue
        parsed = parse_detail_block(block_text)
        results.append((name, parsed))
    return results

In [ ]:
#------------------------------------------------ Begin_Main ----------------------------------------
for k, reg in enumerate(regdict):
    print(f"[INFO] : Working {k + 1}/{len(regdict)} _({reg})_")
    driver.get(regdict[reg])
    sleep(2)

    list_label = Typology[reg]
    reg_ctry, reg_code, list_code = reg.split()

    total_seen = 0
    for cat_url, cotype in CATEGORY_PAGES:
        print(f'  -> Category: {cotype}  ({cat_url})')
        driver.get(cat_url)
        sleep(2)
        entities = extract_entities_from_page(driver.page_source, cat_url)
        print(f'     Found {len(entities)} entities')
        for name, parsed in entities:
            addr = split_address(parsed['address'])
            sqldict['Name'].append(name)
            sqldict['Typology'].append(cotype)
            sqldict['Address_1'].append(addr['address_1'])
            sqldict['Address_2'].append(addr['address_2'])
            sqldict['City'].append(addr['city'])
            sqldict['Zip'].append(addr['zip'])
            sqldict['Cntry'].append(addr['cntry'])
            sqldict['Phone'].append(parsed['phone'])
            sqldict['Fax'].append(parsed['fax'])
            # sqldict['ListLabel'].append(parsed['director'])  # Director General/Representative — keep visible
            sqldict['ListProcessDate'].append(processdate)
            sqldict['RegCtry'].append(reg_ctry)
            sqldict['RegCode'].append(reg_code)
            sqldict['ListCode'].append(list_code)
            sqldict['ListName'].append(list_label)
            # sqldict['ListLanguage'].append('FR')
            sqldict['RegulationType'].append('Regulated')
            sqldict = bourange_same_length_array(sqldict)
            total_seen += 1

    print(f'[INFO] : {reg} — collected {total_seen} entities across {len(CATEGORY_PAGES)} categories')

    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))

[INFO] : Working 1/1 _(DZ BAL 1)_
  -> Category: Commercial Bank  (https://www.bank-of-algeria.dz/banques-commerciales-2/)
     Found 21 entities
  -> Category: Financial Establishment - General Purpose  (https://www.bank-of-algeria.dz/etablissement-financiers-a-vocation-generale/)
     Found 8 entities
  -> Category: Financial Establishment - Specific Purpose  (https://www.bank-of-algeria.dz/etablissements-financiers-a-vocation-specifique/)
     Found 1 entities
  -> Category: Representation Office  (https://www.bank-of-algeria.dz/bureaux-de-representation/)
     Found 6 entities
  -> Category: Banking Association  (https://www.bank-of-algeria.dz/association-des-banques-et-des-etablissements-financiers-abef/)
     Found 1 entities
[INFO] : DZ BAL 1 — collected 37 entities across 5 categories


In [7]:
#------------------------------------------------ Begin_writer ----------------------------------------
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)
print(f'Total rows: {len(df)}')
df.to_excel(filename, 'SQL Ready', index=False)
print(f'Wrote {filename}')
driver.quit()
sleep(3)

Total rows: 37
Wrote DZ BAL SQL Ready 2026-05-27 20.13.00.xlsx


C:\Users\wuj1\AppData\Local\Temp\2\ipykernel_28544\1531189217.py:5: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(filename, 'SQL Ready', index=False)
